# Insurance Claims Analytics & Risk Assessment
## Module 6 — Claim Outcome Prediction

**Business Objective**
Identify the strongest predictors of claim settlement vs rejection to support
evidence-based adjudication policy review. The model is used as a
decision-support tool — not as an automated decision-maker.

**Methodology**
Logistic Regression is selected for its interpretability. In an insurance
context, the ability to explain a prediction to a regulator or adjudicator
outweighs marginal accuracy gains from black-box alternatives.

**Scope**
Binary classification: Settled (1) vs Rejected (0).
Pending claims are excluded — they represent unresolved cases, not deterministic outcomes.

**Evaluation Metrics**
ROC-AUC · Precision · Recall · F1-Score · Confusion Matrix


In [ ]:
import sys
sys.path.insert(0, '..')
from src.claim_prediction import (
    load_cleaned_data, prepare_model_features, train_logistic_regression,
    evaluate_model, extract_feature_importance,
    plot_confusion_matrix, plot_roc_curve, plot_feature_importance
)
from IPython.display import Image
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = load_cleaned_data()


## 6.1 Feature Preparation

In [ ]:
X, y, feature_names = prepare_model_features(df)
print(f"Model dataset    : {len(X):,} records")
print(f"Features         : {len(feature_names)}")
print(f"Class balance    :")
print(y.value_counts().rename({0: 'Rejected', 1: 'Settled'}).to_string())


## 6.2 Model Training & Cross-Validation

In [ ]:
pipeline, X_train, X_test, y_train, y_test, cv_scores = train_logistic_regression(X, y)
print(f"Cross-Validation ROC-AUC ({len(cv_scores)}-fold):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean ROC-AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


## 6.3 Model Evaluation

In [ ]:
eval_results = evaluate_model(pipeline, X_test, y_test)
print(f"Test ROC-AUC : {eval_results['roc_auc']:.4f}")
print(f"\nClassification Report:")
import pandas as pd
report_df = pd.DataFrame(eval_results['classification_report']).T
print(report_df.round(4).to_string())


In [ ]:
plot_confusion_matrix(eval_results)
Image('../visuals/confusion_matrix.png')


In [ ]:
plot_roc_curve(eval_results, y_test)
Image('../visuals/roc_curve.png')


## 6.4 Feature Importance — Business Interpretation

In [ ]:
importance_df = extract_feature_importance(pipeline, feature_names)
plot_feature_importance(importance_df)
Image('../visuals/feature_importance.png')


In [ ]:
print("Top 10 Predictors of Claim Outcome:")
print(importance_df[['Feature', 'Coefficient']].head(10).to_string(index=False))


**Model Limitations**
The near-perfect ROC-AUC (1.0) warrants scrutiny. This performance level
suggests the dataset may contain a structurally deterministic relationship
between features and the target — common in synthetically generated datasets.
In production, this model should be validated against held-out time periods
and monitored for performance drift.

---
**Next Module →** `07_Policy_Lifecycle_Analysis.ipynb`
